In [1]:
import json
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# 1. Load the raw numbers
with open("model_params.json", "r") as f:
    params = json.load(f)

# 2. Create a brand new, empty pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

pipe.named_steps['scaler'].mean_ = np.array(params["scaler_mean"])
pipe.named_steps['scaler'].var_ = np.array(params["scaler_var"])
pipe.named_steps['scaler'].scale_ = np.array(params["scaler_scale"])
pipe.named_steps['scaler'].n_features_in_ = len(params["scaler_mean"])

pipe.named_steps['regressor'].coef_ = np.array(params["coef"])
pipe.named_steps['regressor'].intercept_ = params["intercept"]

pipe

Pipeline(steps=[('scaler', StandardScaler()),
                ('regressor', LinearRegression())])

In [3]:
import pandas as pd

df = pd.read_csv("train_data.csv")

df.head()

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,target,datapointID,subtaskID
0,38.386698,96.119755,74.799984,60.720818,17.224338,16.089440,7.296285,88.208061,60.737244,71.793142,-3.316884,0,1
1,7.509907,103.118333,92.599548,26.231124,27.665694,21.204378,39.120991,61.771641,46.851906,34.885306,-14.268649,1,1
2,57.722605,10.057386,23.272128,33.462029,39.583413,76.698472,14.443321,45.518765,56.918328,0.984856,-20.099994,2,1
3,62.298347,18.787687,9.154591,96.303772,99.248860,81.650804,32.924315,12.399853,69.459085,45.647168,-11.175629,3,1
4,14.072725,51.618305,6.646087,92.645215,29.129085,67.234057,34.152586,55.193712,55.924881,20.460944,-21.376248,4,1


In [8]:
#varianta dummy - facem cate un model stramb gen pt fiecare d asta

features = [col for col in df.columns if col[0] == 'f']
target = df["target"]

df_nou = pd.DataFrame({
    "datapointID": df.datapointID,
    "subtaskID": 1
})

for col in features:
    model = LinearRegression()
    model.fit(pd.DataFrame(target), df[col])
    df_nou[col] = model.predict(pd.DataFrame(target))

df_nou.head()

,datapointID,subtaskID,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10
0,0,1,49.087873,53.840374,53.877345,51.344056,54.836613,52.961309,51.951123,51.044818,52.170555,51.791616
1,1,1,42.765062,45.249851,41.363707,45.240874,40.410757,49.795346,40.938441,39.915599,44.936061,43.846154
2,2,1,39.398437,40.675765,34.700731,41.991190,32.729607,48.109606,35.074660,33.989766,41.084003,39.615536
3,3,1,44.550764,47.676004,44.897834,46.964547,44.484937,50.689483,44.048665,43.058736,46.979242,46.090128
4,4,1,38.661614,39.674676,33.242465,41.279961,31.048503,47.740663,33.791307,32.692833,40.240938,38.689619


In [15]:
answers = []

for idx, row in df_nou.iterrows():
    vals = []
    for col in features:
        vals.append(row[col])
    vals_new = [str(val) for val in vals]
    answers.append(','.join(vals_new))

answers[0]

'49.08787281006223,53.8403738112055,53.87734487642353,51.344056495872394,54.83661307586098,52.961308741207425,51.951122510706924,51.04481761861626,52.170554744111556,51.79161607408959'

In [16]:
df_nou = pd.DataFrame({
    "datapointID": df.datapointID,
    "subtaskID": 1,
    "answer": answers
})

df_nou

,datapointID,subtaskID,answer
0,0,1,"49.08787281006223,53.8403738112055,53.87734487..."
1,1,1,"42.765062352228696,45.24985079822153,41.363707..."
2,2,1,"39.39843658589682,40.675765221142825,34.700730..."
3,3,1,"44.55076352373117,47.67600402713876,44.8978344..."
4,4,1,"38.66161356000666,39.67467635368138,33.2424653..."
...,...,...,...
495,495,1,"44.930426913587155,48.19183588853902,45.649236..."
496,496,1,"51.271579628715585,56.807279715526946,58.19917..."
497,497,1,"38.573459091849315,39.55490477544835,33.067996..."
498,498,1,"56.10190497397561,63.37002919463657,67.7589974..."


In [17]:
df_nou.to_csv("submission.csv", index=False)